# 1. Data Wrangling

Prepare raw TED data for analysis:
- Load raw Excel data
- Remove award-level columns
- Show missing value distribution and drop columns with >40% missing
- Deduplicate and save wrangled data

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(globals()["__vsc_ipynb_file__"]).parent.parent
RAW_PATH = ROOT / "data" / "raw" / "TED award 2018-2023.xlsx"
INTERIM_PATH = ROOT / "data" / "interim" / "ted_awards_wrangled.csv"

In [ ]:
# Load raw data and deduplicate on full columns
df = pd.read_excel(RAW_PATH)
print("Raw shape:", df.shape)

n_dupes = int(df.duplicated().sum())
if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Removed {n_dupes} truly duplicate rows")
else:
    print("No duplicate rows found")
print("Shape after dedup:", df.shape)
df.head()

In [ ]:
AWARD_LEVEL_COLUMNS = [
    "ID_AWARD",
    "ID_LOT_AWARDED",
    "INFO_ON_NON_AWARD",
    "INFO_UNPUBLISHED",
    "B_AWARDED_TO_A_GROUP",
    "WIN_NAME",
    "WIN_NATIONALID",
    "WIN_ADDRESS",
    "WIN_TOWN",
    "WIN_POSTAL_CODE",
    "WIN_COUNTRY_CODE",
    "B_CONTRACTOR_SME",
    "CONTRACT_NUMBER",
    "TITLE",
    "NUMBER_TENDERS_SME",
    "NUMBER_TENDERS_OTHER_EU",
    "NUMBER_TENDERS_NON_EU",
    "NUMBER_OFFERS_ELECTR",
    "AWARD_EST_VALUE_EURO",
    "AWARD_VALUE_EURO",
    "AWARD_VALUE_EURO_FIN_1",
    "B_SUBCONTRACTED",
    "DT_AWARD",
    "ID_NOTICE_CAN",
    "CAE_NATIONALID",
]

# Remove award-level columns
cols_to_drop = [c for c in AWARD_LEVEL_COLUMNS if c in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} award-level columns")
print("Remaining shape:", df.shape)

In [ ]:
# Missing value distribution across all features
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary.sort_values("missing_pct", ascending=False)

print("Missing values per column:")
print(missing_summary[missing_summary["missing_count"] > 0].to_string())

# Drop columns with more than 40% missing
threshold = 40
high_missing = missing_pct[missing_pct > threshold].index.tolist()
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"\nDropped {len(high_missing)} columns with >{threshold}% missing: {high_missing}")
else:
    print(f"\nNo columns exceed {threshold}% missing threshold")
print("Shape after missing filter:", df.shape)

In [ ]:
# Remove rows where NUMBER_OFFERS is missing
n_before = len(df)
df = df.dropna(subset=["NUMBER_OFFERS"]).reset_index(drop=True)
print(f"Removed {n_before - len(df)} rows with missing NUMBER_OFFERS")
print("Shape:", df.shape)

In [ ]:
# Collapse rows that became identical after dropping award-level columns
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Collapsed {n_before - len(df)} rows that shared the same notice-level values")
print("Final shape:", df.shape)

In [ ]:
# Save wrangled data
INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_PATH, index=False)
print("Saved:", INTERIM_PATH)
df.head()